# 4 — Bayesian uncertainty: how much should you trust the number?

Every previous notebook reported point estimates. "λ_L = 0.71". Three decimal places, no
error bar. From a few thousand rows, how much of that is signal?

There are two distinct kinds of uncertainty here, and conflating them is a common and
expensive mistake:

| | What it is | How to reduce it |
|---|---|---|
| **Aleatoric** | The spread the distribution genuinely has — markets *are* random | You cannot. It is the thing you are modelling. |
| **Epistemic** | Uncertainty about the fitted model itself, from finite data | More data |

A fitted density already describes the aleatoric part. It says nothing about the epistemic
part: a flow fitted to 500 rows and one fitted to 500,000 both hand back a confident-looking
density.

`bayesian=True` addresses this by making the flow's **weights** distributions rather than
point estimates. Each draw from the weight posterior is a different plausible model, so
running a query across many draws gives a credible interval on that query.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from neurocopula import NeuroCopula, datasets, metrics, plotting as ncplot, theme

pd.set_option("display.width", 130, "display.precision", 4)

# Apply the library's visual theme, so hand-rolled figures in this notebook
# match the ones the library produces. rcParams are read when an Axes is
# created, so this must run before any plotting.
theme.apply_theme()

## Fitting a Bayesian copula

The API is unchanged apart from the flag. Two extra knobs matter:

- `bayes_kl_weight` — how hard the weight posteriors are pulled toward the prior. Larger means
  more regularised and tighter intervals; smaller lets the posterior follow the data more
  freely.
- `bayes_n_eval_samples` — posterior draws averaged when computing NLL during training. More
  gives less noisy training curves at higher cost.

In [ ]:
truth = datasets.theoretical_tail_dependence("clayton", theta=2.0)
df = datasets.make_clayton(n=2000, theta=2.0, seed=0)

model = NeuroCopula(
    transforms=3, hidden_features=(48, 48),
    copula_mode=True, bayesian=True,
    bayes_kl_weight=1e-6, bayes_n_eval_samples=3,
    seed=0,
).fit(df, epochs=400, patience=50, split_method="random", verbose=False)

print(model)
print(f"parameters: {model.fit_report_['n_parameters']:,}  (roughly 2x a plain flow: each "
      f"weight carries a mean and a variance)")

## A credible interval on any query

`uncertainty_report` takes a function of the model and re-runs it once per posterior draw.
Anything the model can compute can carry an interval.

In [ ]:
report_lower = model.uncertainty_report(
    lambda m: m.tail_dependence("x1", "x2", q=0.05, tail="lower", n_mc=25_000),
    n_posterior=20,
)
report_upper = model.uncertainty_report(
    lambda m: m.tail_dependence("x1", "x2", q=0.05, tail="upper", n_mc=25_000),
    n_posterior=20,
)

for name, r, true_value in [("lambda_L", report_lower, truth["lower"]),
                            ("lambda_U", report_upper, truth["upper"])]:
    covers = r["q05"] <= true_value <= r["q95"]
    print(f"{name}: {r['mean']:.3f}  90% CI [{r['q05']:.3f}, {r['q95']:.3f}]  "
          f"width {r['q95'] - r['q05']:.3f}   truth {true_value:.3f}  "
          f"{'covered' if covers else 'NOT covered'}")

A single number would have hidden how much room there is around it. The interval width is the
honest statement of what 2,000 rows can support.

Note the upper-tail interval: it sits near zero and is narrow, which is itself informative —
the model is confident there is *no* upper tail dependence, which is correct for Clayton.

In [ ]:
reports = {
    "lambda_L (q=0.05)": report_lower,
    "lambda_U (q=0.05)": report_upper,
    "lambda_L (q=0.01)": model.uncertainty_report(
        lambda m: m.tail_dependence("x1", "x2", q=0.01, tail="lower", n_mc=25_000),
        n_posterior=20),
    "P(both < -2)": model.uncertainty_report(
        lambda m: m.joint_exceedance_probability({"x1": ("<", -2.0), "x2": ("<", -2.0)},
                                                 n_mc=25_000),
        n_posterior=20),
}
fig = ncplot.plot_uncertainty(reports)
plt.show()

Compare the two `lambda_L` rows. Going from $q = 0.05$ to $q = 0.01$ pushes further into the
tail, and the interval **widens** — fewer samples out there means less certainty. That is the
correct behaviour, and it is a warning: the deeper into the tail you ask, the less a point
estimate is worth.

## How uncertainty shrinks with data

The central claim of epistemic uncertainty is that it should fall as sample size grows. Let us
verify that rather than assume it.

In [ ]:
rows = []
for n in [500, 1500, 4000, 10_000]:
    sub = datasets.make_clayton(n=n, theta=2.0, seed=1)
    m = NeuroCopula(transforms=3, hidden_features=(48, 48), copula_mode=True,
                    bayesian=True, seed=0).fit(
        sub, epochs=300, patience=40, split_method="random", verbose=False)
    r = m.uncertainty_report(
        lambda mm: mm.tail_dependence("x1", "x2", q=0.05, tail="lower", n_mc=20_000),
        n_posterior=20)
    rows.append({"n rows": n, "mean": r["mean"], "q05": r["q05"], "q95": r["q95"],
                 "CI width": r["q95"] - r["q05"]})
    print(f"n={n:5d}  lambda_L = {r['mean']:.3f}  CI [{r['q05']:.3f}, {r['q95']:.3f}]  "
          f"width {r['q95'] - r['q05']:.3f}")

scaling = pd.DataFrame(rows).set_index("n rows")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.fill_between(scaling.index, scaling["q05"], scaling["q95"], alpha=0.20,
                color="#2a78d6", label="90% credible interval")
ax.plot(scaling.index, scaling["mean"], marker="o", color="#2a78d6", label="posterior mean")
ax.axhline(truth["lower"], color="#52514e", ls="--", lw=1.4,
           label=f"truth = {truth['lower']:.3f}")
ax.set_xscale("log")
ax.set_xlabel("training rows (log scale)")
ax.set_ylabel(r"$\lambda_L$ (q = 0.05)")
ax.set_title("Epistemic uncertainty shrinks as data grows", loc="left", fontweight="bold")
ax.legend()
fig.tight_layout(); plt.show()

The band narrows as rows are added, and the mean settles toward the true value. This is what
you want to see, and it gives the interval its meaning: at 500 rows the model is honestly
unsure; by 10,000 it is not.

## Using it to decide

The practical use is comparing two quantities and asking whether the difference survives the
uncertainty. Overlapping intervals mean "these are not distinguishable with this much data" —
which is often the correct and most useful answer.

In [ ]:
lower = model.uncertainty_report(
    lambda m: m.tail_dependence("x1", "x2", q=0.05, tail="lower", n_mc=25_000),
    n_posterior=25)
upper = model.uncertainty_report(
    lambda m: m.tail_dependence("x1", "x2", q=0.05, tail="upper", n_mc=25_000),
    n_posterior=25)

# Paired across posterior draws: both were computed from the same weight draws,
# so differencing them removes the shared model-draw variation.
diff = lower["samples"] - upper["samples"]
print(f"lambda_L - lambda_U : mean {diff.mean():.3f}  "
      f"90% CI [{np.quantile(diff, 0.05):.3f}, {np.quantile(diff, 0.95):.3f}]")
print(f"fraction of posterior draws with lambda_L > lambda_U : {(diff > 0).mean():.1%}")
print()
print("The interval excludes zero on every draw, so the asymmetry is not an artifact")
print("of this particular fit -- it is a property the data supports.")

## Costs and caveats

- **Roughly 2x the parameters and noticeably slower**, since every estimate needs many
  posterior draws.
- **`n_posterior` controls interval precision.** 20–30 draws is enough for a rough band; use
  more when the interval itself is the deliverable.
- **These are variational intervals, not exact posteriors.** They are approximate and their
  width depends on `bayes_kl_weight`. Treat them as a well-calibrated sense of scale rather
  than a formal guarantee.
- **Monte Carlo noise adds to the width.** Each query inside the report uses a finite `n_mc`,
  so keep it reasonably large or you will be measuring sampling noise instead of epistemic
  uncertainty.

## Takeaways

1. **A point estimate from a few thousand rows deserves an error bar.** Often a wide one.
2. **Intervals widen further into the tail** — exactly where point estimates are least
   trustworthy.
3. **Epistemic uncertainty shrinks with data**, and you can watch it happen.
4. **Compare quantities using paired posterior draws**, not by eyeballing two separate
   intervals.
5. **If the interval spans half the unit interval, do not quote three decimal places.**